# BaseAttentive: Hybrid vs Transformer Architectures

This notebook compares the two main architectural choices in BaseAttentive:
- **Hybrid**: Multi-scale LSTM + Attention (captures temporal patterns + global context)
- **Transformer**: Pure self-attention (global context only)

In [ ]:
import numpy as np

from base_attentive import BaseAttentive

## Common Configuration

In [ ]:
# Shared parameters
STATIC_DIM = 4
DYNAMIC_DIM = 8
FUTURE_DIM = 6
OUTPUT_DIM = 1
FORECAST_HORIZON = 24

shared_params = {
    "static_input_dim": STATIC_DIM,
    "dynamic_input_dim": DYNAMIC_DIM,
    "future_input_dim": FUTURE_DIM,
    "output_dim": OUTPUT_DIM,
    "forecast_horizon": FORECAST_HORIZON,
    "embed_dim": 32,
    "attention_units": 64,
    "num_heads": 8,
    "dropout_rate": 0.1,
}

print("✅ Shared parameters defined")

## Model 1: Hybrid Architecture (LSTM + Attention)

**Characteristics:**
- Multi-scale LSTM for hierarchical temporal feature extraction
- Attention mechanisms for global context
- **Best for**: Complex temporal patterns, shorter sequences
- **Pros**: Captures temporal correlations well
- **Cons**: Slower training/inference

In [ ]:
# Hybrid model: LSTM + Attention
hybrid_config = {
    **shared_params,
    "objective": "hybrid",  # Uses LSTM encoder
    "name": "HybridModel",
}

hybrid_model = BaseAttentive(**hybrid_config)

print("🔧 Hybrid Model Configuration:")
print(f"  Objective: {hybrid_model.objective}")
print(f"  LSTM Units: {hybrid_model.lstm_units}")
print(f"  Scales: {hybrid_model.scales}")
print(f"  Mode: {hybrid_model.mode}")

## Model 2: Transformer Architecture (Pure Attention)

**Characteristics:**
- Pure self-attention stack, no recurrence
- Parallel computation across sequence
- **Best for**: Long sequences, global dependencies
- **Pros**: Faster, captures long-range dependencies
- **Cons**: May miss local temporal patterns

In [ ]:
# Transformer model: Pure Attention
transformer_config = {
    **shared_params,
    "objective": "transformer",  # Pure attention encoder
    "num_encoder_layers": 2,  # Number of attention blocks
    "name": "TransformerModel",
}

transformer_model = BaseAttentive(**transformer_config)

print("🔧 Transformer Model Configuration:")
print(f"  Objective: {transformer_model.objective}")
print(
    f"  Encoder Layers: {transformer_model.num_encoder_layers}"
)
print(
    f"  Attention Units: {transformer_model.attention_units}"
)
print(f"  Mode: {transformer_model.mode}")

## Comparison Table

In [ ]:
import pandas as pd

comparison_data = {
    "Aspect": [
        "Encoder Type",
        "Speed (Training)",
        "Speed (Inference)",
        "Temporal Patterns",
        "Long-Range Dependencies",
        "Memory Usage",
        "Best For",
        "Sequence Length",
    ],
    "Hybrid (LSTM+Attn)": [
        "Multi-scale LSTM",
        "Slower",
        "Slower",
        "Excellent",
        "Good",
        "Higher",
        "Complex temporal data",
        "Short to medium",
    ],
    "Transformer (Pure Attn)": [
        "Self-Attention Stack",
        "Faster",
        "Faster",
        "Good",
        "Excellent",
        "Lower",
        "Long sequences",
        "Long",
    ],
}

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))

## Advanced Configuration: Using architecture_config

You can also use `architecture_config` for fine-grained control.

In [ ]:
# Advanced configuration with architecture_config
custom_architecture = {
    "encoder_type": "transformer",
    "decoder_attention_stack": [
        "cross",
        "hierarchical",
    ],  # Skip memory attention
    "feature_processing": "dense",  # Use dense instead of VSN
}

custom_model = BaseAttentive(
    **shared_params,
    architecture_config=custom_architecture,
    name="CustomModel",
)

print("🎨 Custom Architecture Configuration:")
print(
    f"  Encoder Type: {custom_model.architecture_config.get('encoder_type')}"
)
print(
    f"  Attention Stack: {custom_model.architecture_config.get('decoder_attention_stack')}"
)
print(
    f"  Feature Processing: {custom_model.architecture_config.get('feature_processing')}"
)

## Recommendation

**Use Hybrid when:**
- Your data has strong short-term temporal correlations
- Sequences are short to medium length (< 50 steps)
- You need to capture multi-scale temporal patterns

**Use Transformer when:**
- Long-range dependencies are important
- Sequences are long (> 100 steps)
- You prioritize speed and have sufficient compute
- Your data is mostly stationary with trend/seasonality